In [1]:

!pip install -q streamlit google-generativeai chromadb "youtube-transcript-api==1.2.4" yt-dlp sentence-transformers
!npm install localtunnel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/2

In [2]:
%%writefile app.py
import os
import time
import streamlit as st
from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp
import google.generativeai as genai
import chromadb
from sentence_transformers import SentenceTransformer

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)


@st.cache_resource
def load_embedder():
    return SentenceTransformer('all-MiniLM-L6-v2')

embedding_model = load_embedder()


chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(name="youtube_channel_minilm")



def get_channel_video_ids(channel_url: str) -> list[str]:
    ydl_opts = {
        'extract_flat': True,
        'quiet': True,
        'playlistend': 100
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)
        return [entry['id'] for entry in info['entries']]

def get_transcript_data(video_id: str):
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript_list = ytt_api.list(video_id)

        try:
            transcript = transcript_list.find_transcript(['en', 'en-US', 'en-GB', 'en-CA'])
        except:
            transcript = transcript_list.find_generated_transcript(['en'])

        fetched_transcript = transcript.fetch()
        return fetched_transcript.to_raw_data()

    except Exception as e:
        return f"ERROR: {str(e)}"

def chunk_transcript_with_timestamps(fetched_transcript: list, chunk_size: int = 500) -> list[dict]:
    words = []
    word_timestamps = []

    for item in fetched_transcript:
        item_words = item['text'].split()
        words.extend(item_words)
        word_timestamps.extend([item['start']] * len(item_words))

    chunks = []
    overlap = 50
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        if not chunk_words: continue

        chunk_text = " ".join(chunk_words)
        start_time = int(word_timestamps[i])

        chunks.append({
            "text": chunk_text,
            "start_time": start_time
        })
    return chunks

def index_channel(channel_url: str):
    st.write("Fetching video IDs from channel...")
    video_ids = get_channel_video_ids(channel_url)

    if not video_ids:
        st.error("No videos found.")
        return

    st.write(f"Found {len(video_ids)} videos. Starting local indexing...")

    indexed = 0
    progress_bar = st.progress(0)
    error_log = []

    for idx, video_id in enumerate(video_ids):
        progress_bar.progress((idx + 1) / len(video_ids), text=f"Processing video {idx+1}...")

        transcript_data = get_transcript_data(video_id)

        if isinstance(transcript_data, str) and transcript_data.startswith("ERROR"):
            error_log.append(f"Video {video_id}: {transcript_data}")
            continue

        chunks_data = chunk_transcript_with_timestamps(transcript_data)
        if not chunks_data:
            continue


        chunk_texts = [chunk["text"] for chunk in chunks_data]
        ids = [f"{video_id}_chunk_{i}" for i in range(len(chunks_data))]
        metadatas = [{"video_id": video_id, "url": f"https://youtube.com/watch?v={video_id}&t={chunk['start_time']}s"} for chunk in chunks_data]


        embeddings = embedding_model.encode(chunk_texts).tolist()

        collection.add(
            documents=chunk_texts,
            embeddings=embeddings,
            ids=ids,
            metadatas=metadatas
        )
        indexed += 1
        time.sleep(1)

    if indexed == 0:
        st.error("Failed to index videos.")
        for err in error_log: st.code(err)
    else:
        st.success(f"Indexed {indexed} videos successfully!")
        if error_log:
            st.warning("Some videos failed (usually Shorts with no speaking):")
            for err in error_log: st.write(err)

def query_channel(question: str) -> dict:

    q_embedding = embedding_model.encode(question).tolist()


    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=5
    )

    if not results["documents"][0]:
         return {"answer": "No relevant information found.", "sources": []}

    context = "\n\n".join(results["documents"][0])
    sources = [m["url"] for m in results["metadatas"][0]]


    prompt = f"""Answer the question using ONLY the YouTube transcript context below.
    Include which video(s) the information came from.

    Context:
    {context}

    Question: {question}
    Answer:"""

    model = genai.GenerativeModel('gemini-2.5-flash')
    response = model.generate_content(prompt)

    return {
        "answer": response.text,
        "sources": list(set(sources))
    }


st.title("YouTube RAG via Colab")
st.caption("Using Local MiniLM Embeddings & Gemini Flash!")

st.subheader("1. Index a Channel")
channel_url = st.text_input("Channel URL", placeholder="https://www.youtube.com/@channelname/videos")

if st.button("Index Channel") and channel_url:
    index_channel(channel_url)

st.divider()

st.subheader("2. Ask Questions")
question = st.text_input("Your question")

if st.button("Ask") and question:
    with st.spinner("Searching and generating answer..."):
        result = query_channel(question)
        st.write("### Answer:")
        st.write(result["answer"])
        st.write("### Sources:")
        for url in result["sources"]:
            st.markdown(f"- [{url}]({url})")

Writing app.py


In [3]:
!pip install -q pyngrok

In [4]:
from pyngrok import ngrok
import os

NGROK_AUTH_TOKEN = ""
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


os.environ["GEMINI_API_KEY"] = ""

ngrok.kill()

!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false &>/content/logs.txt &

try:
    public_url = ngrok.connect(8501).public_url
    print(f" Streamlit App is live at: {public_url}")
except Exception as e:
    print(f"Failed to start ngrok tunnel: {e}")

🌟 Your Streamlit App is live at: https://mack-hematogenous-judah.ngrok-free.dev
